In [1]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os
import glob
import cmasher as cmr
import xarray as xr

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *

In [2]:
# Reload edited Python modules automatically before running subsequent cells.
%load_ext autoreload
%autoreload 2

import sys

# Add the project folder only once, even if this cell is rerun.
functions_path = "/scratch/leiff/MCA/amip_clean/project_functions/"
if functions_path not in sys.path:
    sys.path.append(functions_path)

from MCA_cov import *
from significance import *
from plotting import *
from diagnostics import *

In [3]:
# %% 1. Settings and the list of quantities to keep
# Run your existing detrend_monthly_fast definition/import first. These cells call
# that function unchanged; they do not supply a different detrending method.
import numpy as np
import xarray as xr
from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import inspect

if not callable(globals().get("detrend_monthly_fast")):
    raise NameError("Run your detrend_monthly_fast definition/import before these cells.")

# Only this observation-overlap window is processed. Month-start dates below are
# shared labels, not a conversion of model calendars or an interpolation in time.
overlap_data_root = Path("/scratch/leiff/MCA/amip_clean/data")
overlap_start, overlap_end, overlap_ddof = "2000-03", "2014-12", 1
overlap_months = np.arange(np.datetime64(overlap_start, "M"), np.datetime64(overlap_end, "M") + 1)
overlap_month_numbers = overlap_months.astype(int) % 12 + 1
overlap_lat_dim, overlap_lon_dim = "lat", "lon"
overlap_obs_paths = {
    "T": overlap_data_root / "ERA5/tas_194001-202412_g025.nc",
    "R": overlap_data_root / "CERES/CERES_net_200003-202604_g025.nc",
}

# Default weights reproduce your ERA5 first-slice support and cosine(latitude).
# To use exactly your earlier weights instead, set overlap_weights_input = area_weights.
# A local mask can be bool or 0/1, with NaN treated as excluded. It is NOT a weight.
# By default it restricts local maps but leaves T and R global. Setting the final
# switch True also restricts the indices: they then become DOMAIN means, not global.
overlap_weights_input = None
overlap_spatial_mask = None
overlap_global_mean_uses_mask = False

# Names preserve the helper-file order. Historical-matched follows the AMIP order
# and uses already-calculated historical members, not a second round of processing.
overlap_model_names = {
    "amip_hist": np.load(overlap_data_root / "helper/amip_hist_models.npy").astype(str).tolist(),
    "historical": np.load(overlap_data_root / "helper/historical_models.npy").astype(str).tolist(),
}
for experiment, names in overlap_model_names.items():
    if not names or len(names) != len(set(names)):
        raise ValueError(f"{experiment}: model names must be nonempty and unique.")
overlap_model_names["historical_matched"] = [m for m in overlap_model_names["amip_hist"]
                                           if m in overlap_model_names["historical"]]
if not overlap_model_names["historical_matched"]:
    raise ValueError("No historical models overlap with AMIP.")

# ADD METRIC — STEP 1: register its kind, units, and meaning here. This one registry
# initializes member/model/MMM structures and controls their later aggregation.
# All diagnostics and SDs use detrended anomalies. ONLY the two trend_in entries
# retain trends. R denotes the file variable N; its original sign is preserved.
overlap_quantity_info = {
    "corr_Ti_Ri": {"kind": "map", "units": "1", "meaning": "Local T versus local R correlation"},
    "corr_T_Ri": {"kind": "map", "units": "1", "meaning": "Global T versus local R correlation"},
    "corr_Ti_R": {"kind": "map", "units": "1", "meaning": "Local T versus global R correlation"},
    "corr_T_R": {"kind": "scalar", "units": "1", "meaning": "Global T versus global R correlation"},
    "sigma_Ti": {"kind": "map", "units": "T input units", "meaning": "Local temporal T SD"},
    "sigma_Ri": {"kind": "map", "units": "R input units", "meaning": "Local temporal R SD"},
    "sigma_T": {"kind": "scalar", "units": "T input units", "meaning": "Global temporal T SD"},
    "sigma_R": {"kind": "scalar", "units": "R input units", "meaning": "Global temporal R SD"},
    "T_detrended": {"kind": "series", "units": "T input units", "meaning": "Global detrended T anomaly"},
    "R_detrended": {"kind": "series", "units": "R input units", "meaning": "Global detrended R anomaly"},
    "T_trend_in": {"kind": "series", "units": "T input units", "meaning": "Global T anomaly; trend retained"},
    "R_trend_in": {"kind": "series", "units": "R input units", "meaning": "Global R anomaly; trend retained"},

    # Local temperature versus global radiation.
    "cov_Zi_R": {"kind": "map", "units": "W m^-2", "meaning": "Standardized local T versus global R covariance"},
    "cov_Ti_R": {"kind": "map", "units": "W m^-2 K", "meaning": "Local T versus global R covariance"},


    # Global temperature versus local radiation.
    "cov_T_Qi": {"kind": "map", "units": "K", "meaning": "Global T versus standardized local R covariance"},
    "cov_T_Ri": {"kind": "map", "units": "W m^-2 K", "meaning": "Global T versus local R covariance"},

    # Local temperature versus local radiation: either term or both standardized.
    "cov_Zi_Ri": {"kind": "map", "units": "W m^-2", "meaning": "Standardized local T versus local R covariance"},
    "cov_Ti_Qi": {"kind": "map", "units": "K", "meaning": "Local T versus standardized local R covariance"},
    "cov_Ti_Ri": {"kind": "map", "units": "W m^-2 K", "meaning": "Local T versus local R covariance"},
    
    # "reg_T_R": {"kind": "scalar", "units": "R units / T units", "meaning": "Global R on global T slope"},
    # "mean_Ti": {"kind": "map", "units": "T input units", "meaning": "Time mean of RAW local T"},
}
# Follow the ADD METRIC steps in future cells too if adding a new calculation.

In [4]:
# %% 2. Small input helper: check dates, coordinates, dimensions, and labels
def prepare_overlap_field(field, expected_months, reference_lat, reference_lon):
    """Select one monthly field; return (member, time, lat, lon) and source metadata."""
    lat_dim, lon_dim = overlap_lat_dim, overlap_lon_dim
    field = field.sel(time=slice(str(expected_months[0]), str(expected_months[-1])))

    # Compare year/month, not exact days. ERA5, CERES, and model monthly means can
    # use different day-of-month timestamps or calendars without being misaligned.
    month_ids = np.asarray(field.time.dt.year * 12 + field.time.dt.month - 1)
    expected_ids = expected_months.astype(int) + 1970 * 12
    if not np.array_equal(month_ids, expected_ids):
        raise ValueError(f"{field.name}: missing, duplicate, or out-of-order months in the requested window.")
    info = {"variable": field.name, "units": field.attrs.get("units", "not supplied"),
            "long_name": field.attrs.get("long_name", ""),
            "standard_name": field.attrs.get("standard_name", ""),
            "calendar": field.time.encoding.get("calendar", str(field.time.dt.calendar)),
            "source_time": np.asarray([str(t) for t in field.time.values])}

    # We require the comparison grid; nothing here silently regrids, reverses an
    # axis, or rotates longitude. A tiny coordinate-roundoff tolerance is allowed.
    for dim, reference in [(lat_dim, reference_lat), (lon_dim, reference_lon)]:
        coordinate = np.asarray(field[dim])
        if coordinate.ndim != 1 or coordinate.shape != reference.shape:
            raise ValueError(f"{field.name}: {dim} has the wrong shape; regrid explicitly first.")
        if not np.allclose(coordinate, reference, rtol=0., atol=1e-6):
            raise ValueError(f"{field.name}: {dim} does not match the observation grid/order.")

    # A sole extra dimension is the realization dimension. If its coordinate is
    # absent, positions are preserved and explicitly recorded as positional IDs.
    extra_dims = [d for d in field.dims if d not in ("time", lat_dim, lon_dim)]
    if len(extra_dims) > 1:
        raise ValueError(f"Select extra dimensions first: {field.name} has {field.dims}.")
    if extra_dims:
        member_dim = extra_dims[0]
        labelled = member_dim in field.coords
        ids = np.asarray(field[member_dim]) if labelled else np.arange(field.sizes[member_dim])
        info.update(member_dimension=member_dim, member_labels="coordinate" if labelled else "positional")
        field = field.rename({member_dim: "member"}) if member_dim != "member" else field
        field = field.assign_coords(member=ids)
    else:
        field = field.expand_dims(member=["single"])
        info.update(member_dimension=None, member_labels="single field")
    if field.sizes["member"] == 0 or len(np.unique(field.member.values)) != field.sizes["member"]:
        raise ValueError(f"{field.name}: empty or duplicate member labels.")

    # Canonical coordinates allow exact labelled alignment AFTER the checks above.
    field = field.assign_coords(time=expected_months.astype("datetime64[ns]"),
                                **{lat_dim: reference_lat, lon_dim: reference_lon})
    return field.transpose("member", "time", lat_dim, lon_dim), info




In [5]:
# %% 3. Set the common grid/weights and initialize the archive
# Use only ERA5's first selected slice for the grid and default spatial support;
# the analysis loop later checks complete data over ALL months on required cells.
with xr.open_dataset(overlap_obs_paths["T"]) as ds:
    first_T = ds["t2m"].sel(time=slice(overlap_start, overlap_end)).isel(time=0)
    first_T = first_T.transpose(overlap_lat_dim, overlap_lon_dim)
    overlap_lat, overlap_lon = first_T[overlap_lat_dim].values.copy(), first_T[overlap_lon_dim].values.copy()
    reference_grid = first_T.load()

# Labelled user weights/masks must match the grid exactly. Plain arrays must already
# be in (lat, lon) order. There is no implicit coordinate alignment after conversion.
def overlap_grid_array(value):
    if isinstance(value, xr.DataArray):
        value = value.transpose(overlap_lat_dim, overlap_lon_dim)
        xr.align(value, reference_grid, join="exact")
    value = np.asarray(value, dtype=float)
    if value.shape != reference_grid.shape:
        raise ValueError(f"Expected a (lat, lon) array of shape {reference_grid.shape}, got {value.shape}.")
    return value

if overlap_weights_input is None:
    raw_weights = np.cos(np.deg2rad(overlap_lat))[:, None] * np.ones(reference_grid.shape)
    raw_weights = np.where(np.isfinite(reference_grid.values), raw_weights, 0.)
else:
    raw_weights = overlap_grid_array(overlap_weights_input)
if np.any(raw_weights[np.isfinite(raw_weights)] < 0):
    raise ValueError("Area weights cannot be negative.")
weight_support = np.isfinite(raw_weights) & (raw_weights > 0)
if not weight_support.any():
    raise ValueError("No positive finite area weights remain.")
overlap_base_weights = np.where(weight_support, raw_weights, 0.)
overlap_base_weights /= overlap_base_weights.sum()

mask_values = np.ones(reference_grid.shape) if overlap_spatial_mask is None else overlap_grid_array(overlap_spatial_mask)
if np.any(np.isfinite(mask_values) & (mask_values != 0) & (mask_values != 1)):
    raise ValueError("The spatial mask must contain only True/False, 0/1, or NaN, not fractions.")
overlap_local_support = weight_support & np.isfinite(mask_values) & (mask_values == 1)
if not overlap_local_support.any():
    raise ValueError("The spatial mask excludes every local cell.")
overlap_global_weights = overlap_base_weights.copy()
if overlap_global_mean_uses_mask:
    overlap_global_weights = np.where(overlap_local_support, overlap_global_weights, 0.)
    overlap_global_weights /= overlap_global_weights.sum()

# All three groups have the same quantities. Members retain their model key;
# model means will have a leading MODEL axis in overlap_model_names order.
overlap_member_values = {exp: {key: {} for key in overlap_quantity_info} for exp in overlap_model_names}
overlap_model_means = {exp: {} for exp in overlap_model_names}
overlap_MMM = {exp: {} for exp in overlap_model_names}
overlap_model_n_valid = {exp: {} for exp in overlap_model_names}
overlap_MMM_n_valid = {exp: {} for exp in overlap_model_names}
overlap_member_ids = {exp: {} for exp in overlap_model_names}
overlap_sources = {exp: {} for exp in overlap_model_names}
overlap_observations = {}
overlap_coordinates = {"time": overlap_months.astype("datetime64[ns]"), "lat": overlap_lat, "lon": overlap_lon,
                       "month_numbers": overlap_month_numbers, "base_weights": overlap_base_weights,
                       "global_weights": overlap_global_weights, "local_support": overlap_local_support}
overlap_quantity_shapes = {"map": reference_grid.shape, "series": (len(overlap_months),), "scalar": ()}
print(f"{len(overlap_months)} months; grid {reference_grid.shape}; models:",
      {exp: len(names) for exp, names in overlap_model_names.items()})




178 months; grid (72, 144); models: {'amip_hist': 16, 'historical': 53, 'historical_matched': 16}


In [6]:
# %% 4. The calculation for ONE member (also used unchanged for observations)
def calculate_overlap_member(T_raw, R_raw, month_numbers, global_weights, local_support, ddof=1):
    """Return correlations, temporal SDs, and global anomaly series for one member.

    Raw fields: (time, lat, lon). The supplied detrend_monthly_fast removes the
    monthly climatology AND monthwise trend. All moments use the same full window.
    Required cells must be complete; time-varying spatial weights are never used.
    """
    T_raw, R_raw = np.asarray(T_raw, dtype=float), np.asarray(R_raw, dtype=float)
    n_time = len(month_numbers)
    if T_raw.shape != R_raw.shape or T_raw.shape != (n_time,) + global_weights.shape:
        raise ValueError("T/R must share shape (number of months, lat, lon).")
    if not isinstance(ddof, (int, np.integer)) or not 0 <= ddof < n_time:
        raise ValueError("ddof must be an integer from zero to n_time - 1.")
    month_numbers = np.asarray(month_numbers)
    if not np.isin(month_numbers, np.arange(1, 13)).all():
        raise ValueError("month_numbers must be actual calendar-month numbers, 1–12.")
    if any(np.count_nonzero(month_numbers == month) < 3 for month in range(1, 13)):
        raise ValueError("Each calendar month needs at least three years for this detrended workflow.")

    # A local mask need not restrict global indices. Therefore complete data are
    # required on the UNION of the local-map and global-index domains. Do not use
    # nanmean here: varying support would change the meaning of a global time series.
    required = local_support | (global_weights > 0)
    if not np.isfinite(T_raw[:, required]).all() or not np.isfinite(R_raw[:, required]).all():
        raise ValueError("Missing raw data on required cells; choose a fixed common domain explicitly.")
    T_work = np.where(required, T_raw, 0.)
    R_work = np.where(required, R_raw, 0.)
    T_global_raw = np.sum(T_work * global_weights, axis=(1, 2))
    R_global_raw = np.sum(R_work * global_weights, axis=(1, 2))

    # Excluded cells are zero placeholders ONLY while calling your function. This
    # avoids feeding irrelevant NaNs into it. Copies also protect the raw inputs
    # if your function edits arrays in place. Excluded output maps become NaN below.
    Ti = np.asarray(detrend_monthly_fast(T_work.copy()), dtype=float)
    Ri = np.asarray(detrend_monthly_fast(R_work.copy()), dtype=float)
    if Ti.shape != T_raw.shape or Ri.shape != R_raw.shape:
        raise ValueError("detrend_monthly_fast changed the input shape.")
    if not np.isfinite(Ti[:, required]).all() or not np.isfinite(Ri[:, required]).all():
        raise ValueError("detrend_monthly_fast returned nonfinite values on required cells.")
    Ti = np.where(required, Ti - Ti.mean(axis=0), 0.)
    Ri = np.where(required, Ri - Ri.mean(axis=0), 0.)

    # Global detrended series are weighted averages of these same processed fields.
    # This is the convention you already checked against detrending global averages.
    T = np.sum(Ti * global_weights, axis=(1, 2))
    R = np.sum(Ri * global_weights, axis=(1, 2))
    T, R = T - T.mean(), R - R.mean()

    # Trend-retaining anomalies: remove each calendar month's mean, calculated only
    # within this window. March is month 3, not January. No trend is fitted here.
    T_trend_in, R_trend_in = T_global_raw.copy(), R_global_raw.copy()
    for month in range(1, 13):
        selected = month_numbers == month
        T_trend_in[selected] -= T_global_raw[selected].mean()
        R_trend_in[selected] -= R_global_raw[selected].mean()

    # All four sigmas are TEMPORAL SDs, not spatial SDs of a map. Using the same
    # n_time - ddof divisor for each covariance makes the reconstruction exact.
    sigma_Ti, sigma_Ri = Ti.std(axis=0, ddof=ddof), Ri.std(axis=0, ddof=ddof)
    sigma_T, sigma_R = T.std(ddof=ddof), R.std(ddof=ddof)
    cov_Ti_Ri = np.sum(Ti * Ri, axis=0) / (n_time - ddof)
    cov_T_Ri = np.sum(T[:, None, None] * Ri, axis=0) / (n_time - ddof)
    cov_Ti_R = np.sum(Ti * R[:, None, None], axis=0) / (n_time - ddof)
    cov_T_R = np.sum(T * R) / (n_time - ddof)

    # Pearson correlation = covariance / (sigma_A * sigma_B). Undefined constant
    # series remain NaN; they are not interpreted as zero correlation. Area weights
    # enter the global indices above, not an additional weighting of local maps.
    corr_Ti_Ri = np.divide(cov_Ti_Ri, sigma_Ti * sigma_Ri, out=np.full_like(sigma_Ti, np.nan),
                           where=(sigma_Ti > 0) & (sigma_Ri > 0))
    corr_T_Ri = np.divide(cov_T_Ri, sigma_T * sigma_Ri, out=np.full_like(sigma_Ri, np.nan),
                          where=(sigma_T > 0) & (sigma_Ri > 0))
    corr_Ti_R = np.divide(cov_Ti_R, sigma_Ti * sigma_R, out=np.full_like(sigma_Ti, np.nan),
                          where=(sigma_Ti > 0) & (sigma_R > 0))
    corr_T_R = cov_T_R / (sigma_T * sigma_R) if sigma_T > 0 and sigma_R > 0 else np.nan

    # ADD METRIC — STEP 2: calculate it here, while raw and processed fields exist.
    # Use the direct covariance for a slope; a constant R then gives slope zero,
    # whereas a constant predictor T correctly leaves the slope undefined.
    # reg_T_R = cov_T_R / sigma_T**2 if sigma_T > 0 else np.nan
    # mean_Ti = np.where(local_support, T_raw.mean(axis=0), np.nan)

    # Local temperature versus global radiation.
    cov_Zi_R    = np.divide(cov_Ti_R, sigma_Ti, out=np.full_like(sigma_Ti, np.nan), where=(sigma_Ti > 0))
    # Global temperature versus local radiation.
    cov_T_Qi    = np.divide(cov_T_Ri, sigma_Ri, out=np.full_like(sigma_Ri, np.nan), where=(sigma_Ri > 0))
    # Local temperature versus local radiation: either term or both standardized.
    cov_Zi_Ri   = np.divide(cov_Ti_Ri, sigma_Ti, out=np.full_like(sigma_Ti, np.nan), where=(sigma_Ti > 0))
    cov_Ti_Qi   = np.divide(cov_Ti_Ri, sigma_Ri, out=np.full_like(sigma_Ri, np.nan), where=(sigma_Ri > 0))

    # ADD METRIC — STEP 3: add its result here under the key registered in cell 1.
    # No new model/MMM loops or saving code are required for a registered quantity.
    here = {
        "corr_Ti_Ri": np.where(local_support, np.clip(corr_Ti_Ri, -1., 1.), np.nan),
        "corr_T_Ri": np.where(local_support, np.clip(corr_T_Ri, -1., 1.), np.nan),
        "corr_Ti_R": np.where(local_support, np.clip(corr_Ti_R, -1., 1.), np.nan),
        "corr_T_R": np.asarray(np.clip(corr_T_R, -1., 1.)),
        "sigma_Ti": np.where(local_support, sigma_Ti, np.nan),
        "sigma_Ri": np.where(local_support, sigma_Ri, np.nan),
        "sigma_T": np.asarray(sigma_T), "sigma_R": np.asarray(sigma_R),
        "T_detrended": T, "R_detrended": R, "T_trend_in": T_trend_in, "R_trend_in": R_trend_in,
        # Mine
        "cov_Ti_R": np.where(local_support,cov_Ti_R , np.nan),
        "cov_Zi_R": np.where(local_support,cov_Zi_R , np.nan),
        "cov_T_Qi": np.where(local_support,cov_T_Qi , np.nan),
        "cov_T_Ri": np.where(local_support,cov_T_Ri , np.nan),
        "cov_Zi_Ri": np.where(local_support,cov_Zi_Ri , np.nan),
        "cov_Ti_Qi": np.where(local_support,cov_Ti_Qi , np.nan),
        "cov_Ti_Ri": np.where(local_support,cov_Ti_Ri , np.nan),
        # "reg_T_R": np.asarray(reg_T_R), "mean_Ti": mean_Ti,
    }
    return here




In [7]:
# %% 5. Calculate each model/member, then make the separate matched group
# Discover combined files without assuming the historical subfolder's name. Each
# model/experiment/variable must have ONE matching file; ambiguity raises an error.
# If your layout differs, replace only the path selection inside the loop below.
combined_files = list(overlap_data_root.rglob("*_Amon_*combined*_g025.nc"))
experiment_file_tags = {"amip_hist": "amip-hist", "historical": "historical"}

for experiment, file_tag in experiment_file_tags.items():
    for model in overlap_model_names[experiment]:
        paths = {}
        for variable in ["tas", "N"]:
            candidates = [p for p in combined_files if p.name.startswith(f"{variable}_Amon_{model}_")
                          and f"_{file_tag}_" in p.name]
            if len(candidates) != 1:
                raise ValueError(f"{experiment}/{model}/{variable}: expected one file, found {candidates}.")
            paths[variable] = candidates[0]

        # Keep the files open while reading one member at a time. Radiation members
        # are reordered to temperature member LABELS, never paired by position when
        # actual coordinates are available. Numeric coordinates may themselves be
        # only file positions; inspect the saved labels if that applies to your files.
        collected = {key: [] for key in overlap_quantity_info}
        with xr.open_dataset(paths["tas"]) as T_ds, xr.open_dataset(paths["N"]) as R_ds:
            tas, T_info = prepare_overlap_field(T_ds["tas"], overlap_months, overlap_lat, overlap_lon)
            radiation, R_info = prepare_overlap_field(R_ds["N"], overlap_months, overlap_lat, overlap_lon)
            if T_info["member_labels"] != R_info["member_labels"]:
                raise ValueError(f"{model}: T/R member labelling conventions differ; pair explicitly first.")
            if set(tas.member.values) != set(radiation.member.values):
                raise ValueError(f"{model}: temperature and radiation have different member labels.")
            if T_info["member_labels"] == "positional":
                print(f"NOTE: {model}: no member coordinates; using file positions for T/R pairing.")
            radiation = radiation.sel(member=tas.member.values)
            tas, radiation = xr.align(tas, radiation, join="exact")
            overlap_member_ids[experiment][model] = tas.member.values.copy()
            overlap_sources[experiment][model] = {"T": dict(T_info, path=str(paths["tas"])),
                                                  "R": dict(R_info, path=str(paths["N"]))}

            for e, member in enumerate(tas.member.values):
                print(f"{experiment} / {model} / {member} ({e + 1}/{tas.sizes['member']})")
                here = calculate_overlap_member(tas.isel(member=e).values, radiation.isel(member=e).values,
                                                 overlap_month_numbers, overlap_global_weights,
                                                 overlap_local_support, ddof=overlap_ddof)

                # ADD METRIC — STEP 4 is automatic: check the declared keys/shapes,
                # append this member, then stack below. A forgotten registration or
                # assignment fails here instead of silently disappearing from save files.
                if set(here) != set(overlap_quantity_info):
                    raise ValueError("Quantity registry and calculation keys differ; check ADD METRIC steps 1–3.")
                for key, value in here.items():
                    expected_shape = overlap_quantity_shapes[overlap_quantity_info[key]["kind"]]
                    if np.shape(value) != expected_shape:
                        raise ValueError(f"{key}: expected {expected_shape}, got {np.shape(value)}.")
                    collected[key].append(value)

        # Per-model stacks have a MEMBER axis: maps (member,lat,lon), series
        # (member,time), and scalar quantities (member,). Member counts can differ.
        for key in overlap_quantity_info:
            overlap_member_values[experiment][key][model] = np.stack(collected[key], axis=0)

# Historical-matched is a distinct subset, with independent arrays so editing one
# group's values does not edit the full historical archive. No files are read again.
for model in overlap_model_names["historical_matched"]:
    for key in overlap_quantity_info:
        overlap_member_values["historical_matched"][key][model] = overlap_member_values["historical"][key][model].copy()
    overlap_member_ids["historical_matched"][model] = overlap_member_ids["historical"][model].copy()
    overlap_sources["historical_matched"][model] = deepcopy(overlap_sources["historical"][model])




amip_hist / BCC-CSM2-MR / r1i1p1f1 (1/1)
amip_hist / CAMS-CSM1-0 / r1i1p1f1 (1/3)
amip_hist / CAMS-CSM1-0 / r2i1p1f1 (2/3)
amip_hist / CAMS-CSM1-0 / r3i1p1f1 (3/3)
amip_hist / CESM2 / r1i1p1f1 (1/3)
amip_hist / CESM2 / r2i1p1f1 (2/3)
amip_hist / CESM2 / r3i1p1f1 (3/3)
amip_hist / CIESM / r1i1p1f1 (1/3)
amip_hist / CIESM / r2i1p1f1 (2/3)
amip_hist / CIESM / r3i1p1f1 (3/3)
amip_hist / CNRM-CM6-1 / r1i1p1f2 (1/10)
amip_hist / CNRM-CM6-1 / r2i1p1f2 (2/10)
amip_hist / CNRM-CM6-1 / r3i1p1f2 (3/10)
amip_hist / CNRM-CM6-1 / r4i1p1f2 (4/10)
amip_hist / CNRM-CM6-1 / r5i1p1f2 (5/10)
amip_hist / CNRM-CM6-1 / r6i1p1f2 (6/10)
amip_hist / CNRM-CM6-1 / r7i1p1f2 (7/10)
amip_hist / CNRM-CM6-1 / r8i1p1f2 (8/10)
amip_hist / CNRM-CM6-1 / r9i1p1f2 (9/10)
amip_hist / CNRM-CM6-1 / r10i1p1f2 (10/10)
amip_hist / CNRM-CM6-1-HR / r1i1p1f2 (1/1)
amip_hist / CNRM-ESM2-1 / r1i1p1f2 (1/1)
amip_hist / CanESM5 / r1i1p2f1 (1/10)
amip_hist / CanESM5 / r2i1p2f1 (2/10)
amip_hist / CanESM5 / r3i1p2f1 (3/10)
amip_hist / CanE

In [8]:
# %% 6. Observations: the SAME preprocessing/calculations, with no fictitious ensemble
# Select only the two needed variables, so there is no need to drop CERES's others.
with xr.open_dataset(overlap_obs_paths["T"]) as T_ds, xr.open_dataset(overlap_obs_paths["R"]) as R_ds:
    tas, T_info = prepare_overlap_field(T_ds["t2m"], overlap_months, overlap_lat, overlap_lon)
    radiation, R_info = prepare_overlap_field(R_ds["toa_net_all_mon"], overlap_months, overlap_lat, overlap_lon)
    if tas.sizes["member"] != 1 or radiation.sizes["member"] != 1:
        raise ValueError("Expected one ERA5 field and one CERES field, without an ensemble dimension.")
    overlap_observations = calculate_overlap_member(tas.isel(member=0).values, radiation.isel(member=0).values,
                                                    overlap_month_numbers, overlap_global_weights,
                                                    overlap_local_support, ddof=overlap_ddof)
    overlap_sources["observations"] = {"T": dict(T_info, path=str(overlap_obs_paths["T"])),
                                       "R": dict(R_info, path=str(overlap_obs_paths["R"]))}
if set(overlap_observations) != set(overlap_quantity_info):
    raise ValueError("Observation quantities do not match the registry.")




In [9]:
# %% 7. Mean members into models; then mean MODELS into each MMM
def overlap_mean_and_count(values):
    """Arithmetic mean over axis 0, plus the finite contributor count at each entry."""
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    count = finite.sum(axis=0)
    total = np.where(finite, values, 0.).sum(axis=0)
    mean = np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)
    return mean, count

# This averages each MEMBER'S diagnostic. It never correlates ensemble-mean fields
# or takes the SD of an ensemble-mean series. Correlations are arithmetic means
# (not Fisher-z means), matching the descriptive "mean member correlation" target.
# Undefined values are omitted entry-by-entry, and counts make that visible.
for experiment, names in overlap_model_names.items():
    for key in overlap_quantity_info:
        model_values, model_counts = [], []
        for model in names:
            mean, count = overlap_mean_and_count(overlap_member_values[experiment][key][model])
            model_values.append(mean)
            model_counts.append(count)
        overlap_model_means[experiment][key] = np.stack(model_values, axis=0)
        overlap_model_n_valid[experiment][key] = np.stack(model_counts, axis=0)
        overlap_MMM[experiment][key], overlap_MMM_n_valid[experiment][key] = overlap_mean_and_count(
            overlap_model_means[experiment][key])

# The averaging of series is also literal: member mean, then equal-model mean at
# each month. Cancellation in those series is NOT a measure of member variability;
# mean(sigma_R) and std(mean(R_series)) answer different questions.
try:
    detrend_source = inspect.getsource(detrend_monthly_fast)
except (OSError, TypeError):
    detrend_source = None  # Notebook/import source is not always retrievable; record that honestly.
overlap_metadata = {
    "schema_version": 1, "created_utc": datetime.now(timezone.utc).isoformat(),
    "start_month": overlap_start, "end_month": overlap_end, "n_time": len(overlap_months), "ddof": overlap_ddof,
    "detrending_function": "detrend_monthly_fast", "detrending_source": detrend_source,
    "preprocessing": "User's monthwise detrend/deseasonalize; then center through time",
    "trend_in_series": "Subtract each calendar month's mean within the overlap window; no detrending",
    "global_mean_uses_mask": overlap_global_mean_uses_mask,
    "index_domain": "masked domain" if overlap_global_mean_uses_mask else "positive base-weight support",
    "weight_source": "ERA5 first-slice support times cosine(latitude)" if overlap_weights_input is None else "user supplied",
    "missing_data": "Complete raw data required on fixed local/global support; undefined correlations are NaN",
    "aggregation": "Arithmetic member means, then equally weighted finite model means; no Fisher-z transform",
    "radiation_convention": "R is input N / CERES toa_net_all_mon; units and sign unchanged",
    "local_fields_retained": False,
}
print("Archive calculated. Keys:", list(overlap_quantity_info))




Archive calculated. Keys: ['corr_Ti_Ri', 'corr_T_Ri', 'corr_Ti_R', 'corr_T_R', 'sigma_Ti', 'sigma_Ri', 'sigma_T', 'sigma_R', 'T_detrended', 'R_detrended', 'T_trend_in', 'R_trend_in', 'cov_Zi_R', 'cov_Ti_R', 'cov_T_Qi', 'cov_T_Ri', 'cov_Zi_Ri', 'cov_Ti_Qi', 'cov_Ti_Ri']


In [13]:
# %% 8. Quick inspection and reconstruction of the three original member maps
experiment = "historical_matched"
model = overlap_model_names[experiment][0]
member_values = overlap_member_values[experiment]
print("Example:", experiment, model, "member IDs:", overlap_member_ids[experiment][model])
print("Member maps:", member_values["corr_Ti_R"][model].shape)
print("Model-mean maps:", overlap_model_means[experiment]["corr_Ti_R"].shape)
print("MMM series:", overlap_MMM[experiment]["R_detrended"].shape)
print("Observed sigma(R):", overlap_observations["sigma_R"])

# Reconstruct per member FIRST, before calculating model/MMM means. Products of
# mean correlations and mean SDs do not, in general, equal mean covariances.
rho_Ti_R = member_values["corr_Ti_R"][model]
rho_T_Ri = member_values["corr_T_Ri"][model]
rho_Ti_Ri = member_values["corr_Ti_Ri"][model]
sTi, sRi = member_values["sigma_Ti"][model], member_values["sigma_Ri"][model]
sT = member_values["sigma_T"][model][:, None, None]
sR = member_values["sigma_R"][model][:, None, None]
covRTi_reconstructed = rho_Ti_R * sR
covTRi_reconstructed = rho_T_Ri * sT
regTiRi_reconstructed = rho_Ti_Ri * sRi

# A standardized predictor with zero SD is undefined. But a CONSTANT unstandardized
# response gives covariance/slope zero, even though its correlation is undefined.
covRTi_reconstructed = np.where((sTi > 0) & (sR == 0), 0., covRTi_reconstructed)
covTRi_reconstructed = np.where((sRi > 0) & (sT == 0), 0., covTRi_reconstructed)
regTiRi_reconstructed = np.where((sTi > 0) & (sRi == 0), 0., regTiRi_reconstructed)

# These examples do not overwrite your old diagnostic dictionaries. For any NEW
# derived metric, aggregate its reconstructed MEMBER maps using cell 7's pattern.
# With positive SDs, other useful variants include:
# raw Cov(Ti,R) = rho_Ti_R * sTi * sR; Cov(Ti,R/sigma_R) = rho_Ti_R * sTi.




Example: historical_matched BCC-CSM2-MR member IDs: ['r1i1p1f1' 'r2i1p1f1' 'r3i1p1f1']
Member maps: (3, 72, 144)
Model-mean maps: (16, 72, 144)
MMM series: (178,)
Observed sigma(R): 0.6319671715057029


In [10]:
# %% 9. Save: a NEW directory, leaving the older five-map archives untouched
# Run this cell when satisfied with the results. Pick a new directory for another
# mask/window, or deliberately enable overwrite for these specifically named files.
# overlap_data_root = Path("/scratch/leiff/MCA/amip_clean/data")
overlap_save_dir = overlap_data_root / "maps_2000_2014_covImp/overlap_core_200003_201412"
overlap_overwrite = False
overlap_save_items = {
    "overlap_member_values": overlap_member_values, "overlap_model_means": overlap_model_means,
    "overlap_MMM": overlap_MMM, "overlap_observations": overlap_observations,
    "overlap_model_n_valid": overlap_model_n_valid, "overlap_MMM_n_valid": overlap_MMM_n_valid,
    "overlap_model_names": overlap_model_names, "overlap_member_ids": overlap_member_ids,
    "overlap_coordinates": overlap_coordinates, "overlap_quantity_info": overlap_quantity_info,
    "overlap_sources": overlap_sources, "overlap_metadata": overlap_metadata,
}
existing = [overlap_save_dir / f"{name}.npy" for name in overlap_save_items
            if (overlap_save_dir / f"{name}.npy").exists()]
if existing and not overlap_overwrite:
    raise FileExistsError(f"Archive files already exist; choose a new folder or set overlap_overwrite=True: {existing}")
overlap_save_dir.mkdir(parents=True, exist_ok=True)
for name, value in overlap_save_items.items():
    np.save(overlap_save_dir / f"{name}.npy", value, allow_pickle=True)
print(f"Saved {len(overlap_save_items)} dictionaries to {overlap_save_dir}")




Saved 12 dictionaries to /scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412


In [ ]:
# %% 10. Load-only cell: usable in a fresh kernel without rerunning the calculations
# Only load pickle-containing .npy dictionaries that you created/trust.
import numpy as np
from pathlib import Path

stop

overlap_save_dir = Path("/scratch/leiff/MCA/amip_clean/data/overlap_core_200003_201412")
overlap_member_values = np.load(overlap_save_dir / "overlap_member_values.npy", allow_pickle=True).item()
overlap_model_means = np.load(overlap_save_dir / "overlap_model_means.npy", allow_pickle=True).item()
overlap_MMM = np.load(overlap_save_dir / "overlap_MMM.npy", allow_pickle=True).item()
overlap_observations = np.load(overlap_save_dir / "overlap_observations.npy", allow_pickle=True).item()
overlap_model_n_valid = np.load(overlap_save_dir / "overlap_model_n_valid.npy", allow_pickle=True).item()
overlap_MMM_n_valid = np.load(overlap_save_dir / "overlap_MMM_n_valid.npy", allow_pickle=True).item()
overlap_model_names = np.load(overlap_save_dir / "overlap_model_names.npy", allow_pickle=True).item()
overlap_member_ids = np.load(overlap_save_dir / "overlap_member_ids.npy", allow_pickle=True).item()
overlap_coordinates = np.load(overlap_save_dir / "overlap_coordinates.npy", allow_pickle=True).item()
overlap_quantity_info = np.load(overlap_save_dir / "overlap_quantity_info.npy", allow_pickle=True).item()
overlap_sources = np.load(overlap_save_dir / "overlap_sources.npy", allow_pickle=True).item()
overlap_metadata = np.load(overlap_save_dir / "overlap_metadata.npy", allow_pickle=True).item()


In [12]:
np.load(overlap_save_dir / "overlap_quantity_info.npy", allow_pickle=True).item()

{'corr_Ti_Ri': {'kind': 'map',
  'units': '1',
  'meaning': 'Local T versus local R correlation'},
 'corr_T_Ri': {'kind': 'map',
  'units': '1',
  'meaning': 'Global T versus local R correlation'},
 'corr_Ti_R': {'kind': 'map',
  'units': '1',
  'meaning': 'Local T versus global R correlation'},
 'corr_T_R': {'kind': 'scalar',
  'units': '1',
  'meaning': 'Global T versus global R correlation'},
 'sigma_Ti': {'kind': 'map',
  'units': 'T input units',
  'meaning': 'Local temporal T SD'},
 'sigma_Ri': {'kind': 'map',
  'units': 'R input units',
  'meaning': 'Local temporal R SD'},
 'sigma_T': {'kind': 'scalar',
  'units': 'T input units',
  'meaning': 'Global temporal T SD'},
 'sigma_R': {'kind': 'scalar',
  'units': 'R input units',
  'meaning': 'Global temporal R SD'},
 'T_detrended': {'kind': 'series',
  'units': 'T input units',
  'meaning': 'Global detrended T anomaly'},
 'R_detrended': {'kind': 'series',
  'units': 'R input units',
  'meaning': 'Global detrended R anomaly'},
 'T_t